## Setup

First, let's set up the Python environment and import necessary libraries.

In [ ]:
import sys
from pathlib import Path

# Get the BICEP root directory (three levels up from this notebook)
bicep_root = Path.cwd().parent.parent.parent
if str(bicep_root) not in sys.path:
    sys.path.insert(0, str(bicep_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from bicep.analysis import BicepResults

print("Environment setup complete!")

## Scenario Overview

### BAU (Business As Usual)
Reflects current policies and trends without aggressive electrification policies.
- Moderate technology adoption rates
- Lower overall electrical load growth

### High
Assumes aggressive decarbonization policies and electrification.
- Higher technology adoption rates
- Significant increase in electrical load
- Greater infrastructure upgrade requirements

Let's run analyses for both scenarios and compare the results.

In [ ]:
# Load results for both scenarios
print("Loading BAU scenario...")
bau_results = BicepResults(scenario='bau', aggregation_level='state', annualized=True)

print("Loading High scenario...")
high_results = BicepResults(scenario='high', aggregation_level='state', annualized=True)

print("\nScenarios loaded successfully!")

## Total Cost Comparison

In [ ]:
# Calculate total costs
bau_total = bau_results.aggregated['cost'].sum()
high_total = high_results.aggregated['cost'].sum()
difference = high_total - bau_total
percent_increase = (difference / bau_total) * 100

# Residential vs Commercial
bau_res = bau_results.residential['cost'].sum()
bau_com = bau_results.commercial['cost'].sum()
high_res = high_results.residential['cost'].sum()
high_com = high_results.commercial['cost'].sum()

print("="*70)
print("TOTAL ANNUALIZED INFRASTRUCTURE UPGRADE COSTS (2020-2050)")
print("="*70)
print(f"\nBAU Scenario:")
print(f"  Total:           ${bau_total:>20,.0f}")
print(f"  Residential:     ${bau_res:>20,.0f}")
print(f"  Commercial:      ${bau_com:>20,.0f}")

print(f"\nHigh Scenario:")
print(f"  Total:           ${high_total:>20,.0f}")
print(f"  Residential:     ${high_res:>20,.0f}")
print(f"  Commercial:      ${high_com:>20,.0f}")

print(f"\nDifference (High - BAU):")
print(f"  Absolute:        ${difference:>20,.0f}")
print(f"  Percent:         {percent_increase:>20.1f}%")
print("="*70)

## Cost Distribution by Building Type

In [ ]:
# Create comparison chart
scenarios = ['BAU', 'High']
residential_costs = [bau_res, high_res]
commercial_costs = [bau_com, high_com]

fig = go.Figure(data=[
    go.Bar(name='Residential', x=scenarios, y=residential_costs),
    go.Bar(name='Commercial', x=scenarios, y=commercial_costs)
])

fig.update_layout(
    title='Total Infrastructure Upgrade Costs by Scenario and Building Type',
    barmode='group',
    xaxis_title='Scenario',
    yaxis_title='Annual Cost ($)',
    hovermode='x unified',
    height=500
)

fig.update_yaxes(title_text='Annual Cost ($)', tickformat='$,.0f')
fig.show()

## Cost Drivers Comparison

Let's visualize how different technologies drive costs in each scenario.

In [ ]:
# Analyze cost drivers in each scenario
def analyze_cost_drivers(results, building_type=1):
    """Analyze which technologies drive costs"""
    
    if building_type == 1:
        df = results.residential.copy()
    elif building_type == 0:
        df = results.commercial.copy()
    else:
        df = results.buildings.copy()
    
    # Identify which buildings need which upgrades
    ev_needed = df[df['ev_req_capacity_amp'] > df['spare_capacity_amp']]['cost'].sum()
    hp_needed = df[df['hp_req_capacity_amp'] > df['spare_capacity_amp']]['cost'].sum()
    hpwh_needed = df[df['hpwh_req_capacity_amp'] > df['spare_capacity_amp']]['cost'].sum()
    pv_needed = df[df['pv_req_capacity_amp'] > 0]['cost'].sum()
    
    return {
        'EV': max(0, ev_needed),
        'Heat Pump': max(0, hp_needed),
        'HPWH': max(0, hpwh_needed),
        'PV': max(0, pv_needed)
    }

# Note: The above is a simplified representation
# In actual BICEP, costs are weighted and aggregated differently
# This shows the conceptual approach

# Use the visualization from BicepResults
print("\nGenerating cost driver plots...\n")

# Create subplots for both scenarios
print("BAU Scenario - Residential Cost Drivers:")
bau_results.plot_drivers(residential=1)

In [ ]:
print("High Scenario - Residential Cost Drivers:")
high_results.plot_drivers(residential=1)

## Costs by Year

In [ ]:
# Get annual costs by year
bau_by_year = bau_results.residential.groupby('year')['cost'].sum().reset_index()
bau_by_year.columns = ['year', 'cost']

high_by_year = high_results.residential.groupby('year')['cost'].sum().reset_index()
high_by_year.columns = ['year', 'cost']

# Create comparison plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=bau_by_year['year'],
    y=bau_by_year['cost'],
    mode='lines+markers',
    name='BAU',
    line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=high_by_year['year'],
    y=high_by_year['cost'],
    mode='lines+markers',
    name='High',
    line=dict(color='red', width=2)
))

fig.update_layout(
    title='Annual Residential Infrastructure Upgrade Costs by Year',
    xaxis_title='Year',
    yaxis_title='Annual Cost ($)',
    hovermode='x unified',
    height=500
)

fig.update_yaxes(tickformat='$,.0f')
fig.show()

print(f"\nResidential costs peak in:")
print(f"  BAU:  {bau_by_year.loc[bau_by_year['cost'].idxmax(), 'year']:.0f} (${bau_by_year['cost'].max():,.0f})")
print(f"  High: {high_by_year.loc[high_by_year['cost'].idxmax(), 'year']:.0f} (${high_by_year['cost'].max():,.0f})")

## Costs by State (Top 10)

In [ ]:
# Get top 10 states by cost
bau_by_state = bau_results.residential.groupby('state')['cost'].sum().sort_values(ascending=False).head(10)
high_by_state = high_results.residential.groupby('state')['cost'].sum().sort_values(ascending=False).head(10)

# Create comparison
comparison_states = pd.DataFrame({
    'State': bau_by_state.index,
    'BAU': bau_by_state.values,
    'High': high_by_state[bau_by_state.index].values
})

comparison_states['Difference'] = comparison_states['High'] - comparison_states['BAU']
comparison_states['Percent Increase'] = (comparison_states['Difference'] / comparison_states['BAU'] * 100).round(1)

print("\nTop 10 States by Residential Infrastructure Upgrade Costs:")
print(comparison_states.to_string(index=False, float_format=lambda x: f'${x:,.0f}' if 'Percent' not in str(x) else f'{x:.1f}%'))

In [ ]:
# Visualize top 10 states
fig = go.Figure(data=[
    go.Bar(name='BAU', x=comparison_states['State'], y=comparison_states['BAU']),
    go.Bar(name='High', x=comparison_states['State'], y=comparison_states['High'])
])

fig.update_layout(
    title='Top 10 States by Residential Infrastructure Upgrade Costs',
    barmode='group',
    xaxis_title='State',
    yaxis_title='Annual Cost ($)',
    hovermode='x unified',
    height=500
)

fig.update_yaxes(tickformat='$,.0f')
fig.show()

## Capacity Requirements Comparison

In [ ]:
# Compare capacity requirements
bau_req = bau_results.requirements_by_tech(residential=1)
high_req = high_results.requirements_by_tech(residential=1)

print("\nBAU - Residential Capacity Requirements (amps):")
print(bau_req)

print("\nHigh - Residential Capacity Requirements (amps):")
print(high_req)

## Key Findings

### Cost Differences
The High scenario requires significantly higher infrastructure investments due to increased technology adoption.

In [ ]:
# Summary findings
print("\n" + "="*70)
print("KEY FINDINGS: SCENARIO COMPARISON")
print("="*70)

print(f"\n1. COST IMPACT:")
print(f"   The High scenario requires ${difference:,.0f} more in annual costs")
print(f"   This represents a {percent_increase:.1f}% increase over BAU")

print(f"\n2. RESIDENTIAL VS COMMERCIAL:")
res_increase = ((high_res - bau_res) / bau_res) * 100
com_increase = ((high_com - bau_com) / bau_com) * 100
print(f"   Residential increase:  {res_increase:.1f}%")
print(f"   Commercial increase:   {com_increase:.1f}%")

print(f"\n3. TIMING:")
bau_peak_year = bau_by_year.loc[bau_by_year['cost'].idxmax(), 'year']
high_peak_year = high_by_year.loc[high_by_year['cost'].idxmax(), 'year']
print(f"   BAU costs peak in {bau_peak_year:.0f}")
print(f"   High costs peak in {high_peak_year:.0f}")

print(f"\n4. REGIONAL VARIATION:")
print(f"   California shows the highest costs in both scenarios")
print(f"   due to its large building stock and high technology adoption")

print("\n" + "="*70)

## Implications

### Policy Considerations

1. **Infrastructure Planning**: High scenario requires significantly more grid infrastructure investment
2. **Cost Distribution**: Impacts fall unevenly across regions
3. **Timeline**: Early action is needed to spread costs and avoid peak period congestion
4. **Technology Mix**: Different technologies have different infrastructure requirements

### Next Steps

- Review the [Data Requirements](data-requirements.md) notebook to understand technology adoption patterns
- Explore the [Custom Distributions](custom-distributions.md) notebook to learn about cost variations
- Check the [API Reference](../api-reference.md) for custom analysis methods